# 🍔 Food Delivery – AI Agent for Relational Database
**Databases Course · Final Project**

Architecture:
```
User Prompt → LangChain SQL Agent → Gemini LLM → MySQL → Results + Explanation
```

## 0 · Setup & Imports

In [ ]:
# Install dependencies (run once)
# !pip install langchain langchain-community langchain-google-genai \
#              sqlalchemy mysql-connector-python pymysql pandas matplotlib seaborn python-dotenv

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# LangChain
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents.agent_types import AgentType

load_dotenv()

# ── Credentials ───────────────────────────────────────────────
DB_USER     = os.getenv('DB_USER',     'root')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')
DB_HOST     = os.getenv('DB_HOST',     'localhost')
DB_PORT     = os.getenv('DB_PORT',     '3306')
DB_NAME     = os.getenv('DB_NAME',     'food_delivery')
GEMINI_KEY  = os.getenv('GOOGLE_API_KEY', 'YOUR_GEMINI_API_KEY_HERE')  # <-- fill in

print('✅  Imports OK')

## 1 · Database Connection

In [ ]:
# SQLAlchemy engine (used by pandas & LangChain)
engine = create_engine(
    f'mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

with engine.connect() as conn:
    result = conn.execute(text('SELECT COUNT(*) FROM orders'))
    print(f'✅  Connected. Orders in DB: {result.scalar():,}')

## 2 · ETL – Load CSV into MySQL

In [ ]:
import subprocess, sys

# Run the ETL script (adjust path if needed)
ETL_SCRIPT = '../sql/etl_load.py'
CSV_PATH   = '../data/food_delivery_dataset.csv'

result = subprocess.run(
    [sys.executable, ETL_SCRIPT, '--csv', CSV_PATH],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 3 · Exploratory SQL – Quick Sanity Checks

In [ ]:
def query(sql):
    """Run SQL and return a DataFrame."""
    return pd.read_sql(sql, engine)

# Row counts per table
tables = ['locations','cuisines','routes','customers','restaurants',
          'orders','order_items','deliveries','feedback']
counts = {t: query(f'SELECT COUNT(*) AS n FROM {t}').iloc[0,0] for t in tables}
pd.DataFrame(counts.items(), columns=['table','rows'])

## 4 · SQL Insights via Views & Procedures

In [ ]:
# ── Traffic & Weather impact on delays ───────────────────────
df_tw = query('SELECT * FROM vw_traffic_weather_impact ORDER BY avg_delay_min DESC')

pivot = df_tw.pivot(index='weather_condition', columns='traffic_condition', values='avg_delay_min')
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('Average Delivery Delay (min)\nby Traffic × Weather')
plt.tight_layout(); plt.show()

In [ ]:
# ── Loyalty program analysis ──────────────────────────────────
df_loy = query('SELECT * FROM vw_loyalty_analysis')
df_loy['loyalty'] = df_loy['loyalty_program'].map({1: 'Member', 0: 'Non-member'})

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col, label in zip(axes,
                          ['avg_order_value', 'avg_rating', 'avg_satisfaction'],
                          ['Avg Order Value ($)', 'Avg Rating', 'Avg Satisfaction']):
    ax.bar(df_loy['loyalty'], df_loy[col], color=['#2196F3','#FF9800'])
    ax.set_title(label); ax.set_ylim(0, df_loy[col].max() * 1.2)
plt.suptitle('Loyalty Programme Impact', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Route performance ─────────────────────────────────────────
df_rt = query('SELECT * FROM vw_route_performance ORDER BY avg_delay_min')
fig, ax = plt.subplots(figsize=(10, 4))
colors = df_rt['avg_delay_min'].apply(lambda v: '#4CAF50' if v < 18 else '#F44336')
ax.barh(df_rt['route_name'], df_rt['avg_delay_min'], color=colors)
ax.set_xlabel('Average Delay (min)')
ax.set_title('Route Performance – Average Delivery Delay')
plt.tight_layout(); plt.show()

In [ ]:
# ── Stored procedure: delivery stats by method ────────────────
df_method = query('CALL sp_delivery_stats_by_method()')
df_method

In [ ]:
# ── Stored procedure: top 10 food items ──────────────────────
df_food = query('CALL sp_top_food_items(10)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_food['food_item'], df_food['times_ordered'], color='#673AB7')
ax.set_xlabel('Times Ordered')
ax.set_title('Top 10 Most-Ordered Food Items')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# ── Stored procedure: late deliveries (delay > 25 min) ────────
df_late = query('CALL sp_late_deliveries(25)')
print(f'Late deliveries (>25 min): {len(df_late):,}')
df_late.head(10)

## 5 · LangChain + Gemini — AI Text-to-SQL Agent

> **Architecture**
> ```
> User Prompt
>      ↓
> LangChain SQL Agent   ← schema introspection
>      ↓
> Gemini (google-genai)  ← generates SQL
>      ↓
> MySQL                  ← executes query
>      ↓
> Result + Explanation
> ```

In [ ]:
# ── 5.1  Connect LangChain to the database ────────────────────
db = SQLDatabase.from_uri(
    f'mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    include_tables=[
        'customers','locations','cuisines','restaurants',
        'orders','order_items','deliveries','routes','feedback'
    ],
    sample_rows_in_table_info=2,
)
print(db.get_usable_table_names())

In [ ]:
# ── 5.2  Initialise Gemini LLM ────────────────────────────────
# Fill GOOGLE_API_KEY in .env or replace the string below
llm = ChatGoogleGenerativeAI(
    model='gemini-1.5-flash',
    google_api_key=GEMINI_KEY,
    temperature=0,
)
print('✅  Gemini LLM ready')

In [ ]:
# ── 5.3  Build the SQL Agent ──────────────────────────────────
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True,
)
print('✅  SQL Agent ready')

In [ ]:
# ── 5.4  Helper: ask the agent ────────────────────────────────
def ask(question: str):
    print(f'\n🔍  Question: {question}')
    print('─' * 60)
    response = agent.invoke({'input': question})
    print('\n💡  Answer:')
    print(response['output'])
    return response

In [ ]:
# ── 5.5  Demo Queries ─────────────────────────────────────────
ask('Which city has the highest average delivery delay?')

In [ ]:
ask('What delivery method has the best average customer rating, and how does it compare to the others?')

In [ ]:
ask('Do loyalty program members spend more per order on average? By how much?')

In [ ]:
ask('Which weather and traffic conditions cause the most delays?')

In [ ]:
ask('What are the top 5 restaurants by total revenue?')

## 6 · Interactive Chat with the Agent

In [ ]:
# Run this cell to start an interactive loop (type 'quit' to exit)
while True:
    user_input = input('\nAsk the database (or type quit): ').strip()
    if user_input.lower() in ('quit', 'exit', 'q', ''):
        print('Goodbye!')
        break
    ask(user_input)

## 7 · Problem Statement Analysis

Below are targeted SQL analyses addressing the four project problem statements.

In [ ]:
# Problem 1 – What factors most increase delivery delay?
ask('Analyse which factors (weather, traffic, delivery method, route type) most strongly correlate with high delivery delay. Rank them.')

In [ ]:
# Problem 2 – Does route efficiency actually reduce delays?
ask('Compare average delivery delay for routes with route_efficiency above 0.8 vs below 0.6. Is there a meaningful difference?')

In [ ]:
# Problem 3 – Customer satisfaction drivers
ask('What combination of food_temperature, food_freshness and packaging_quality gives the highest average customer_satisfaction?')

In [ ]:
# Problem 4 – Loyalty programme ROI
ask('Calculate total revenue from loyalty members vs non-members. What percentage of total revenue comes from loyalty members?')